# Goal: $\sigma$ from $0.05$ to $2$, through $L = 8$, before Friday

This notebook is the plan. It does not run the projector.

Today is Sunday 25 October 2026. The figure has to be ready before Friday 30 October. New runs stop on Thursday 29 October so the plot can be made that day.

The graph is the deliverable, and it is the $L = 8$ torus only, 192 sites. It shows $[\langle q^{2}\rangle]$ and $[B]$ against $\Gamma$, with one line for each $\sigma = 0.05, 0.1, 0.2, 0.5, 1, 2$, 32 draws, and jackknife errors. Plotted against $\Gamma$, the drop moves toward the origin as $\sigma$ gets smaller. Plotted against $\Gamma/\sigma$, the lines fall on top of each other.

The computation that runs first is a trial, not that figure. One periodic sample at $L = 6$ is projected at $\sigma = 0.05, 1, 2$ and $\Gamma/\sigma = 1, 2, 3$, with the Python projector and the $1/\sigma$ projection time. The trial is read before any $L = 8$ production starts. $L = 4$ and a full $L = 6$ grid are not part of this deadline.

The question is how the disorder-averaged overlap moves when the standard deviation of the Gaussian couplings is changed, at $h = 0$ on the periodic Kagome torus. That standard deviation is $\sigma$. The couplings are $J_b = \sigma z_b$, with one stored standard normal $z_b$ per bond. The operator is

$$
H = -\sum_b J_b \sigma_i^z \sigma_j^z - \Gamma \sum_i \sigma_i^x.
$$

Rescaling gives $H(\sigma, \Gamma) = \sigma\, H(1, \Gamma/\sigma)$. Every dimensionless observable depends on the transverse field only through $\Gamma/\sigma$. The transition sits at

$$
\Gamma_c(\sigma) = \sigma\, \Gamma_c(1).
$$

A larger $\sigma$ moves the drop in $q_{\mathrm{EA}}^{2}$ to larger $\Gamma$. Plotted against $\Gamma/\sigma$, the curves fall on top of each other. That collapse is the result. A failure to collapse is a bug in the scaling or in the projector. Adding noise to a nonzero mean coupling is a different model and is outside this plan.


## Observable

Two replicas share one draw of $J$ and do not interact. The equal-time overlap is

$$
q = N^{-1} \sum_i s_i^{(a)} s_i^{(b)}.
$$

The Edwards–Anderson signal is the disorder average of the pure second moment, $q_{\mathrm{EA}}^{2} = [\langle q^{2}\rangle]$. The Binder ratio is formed on each sample and then averaged,

$$
B = \frac{1}{2}\left(3 - \frac{\langle q^{4}\rangle}{\langle q^{2}\rangle^{2}}\right).
$$

The walkers carry the ground-state amplitude, so the overlap is the forward-walked tag from `kagome_qea.pqmc.project`, not a snapshot and not the one-copy energy run. Each walker starts as an independent array of $+1$ and $-1$, with shape `(2, N)`. The guiding function is $\psi = 1$. The first 40 blocks, a projection time $\tau = 8$, are discarded.

The plotted error is the jackknife error across the 32 draws. One forward-walked series is not marked resolved. A Binder crossing is a window in $\Gamma/\sigma$, not a quoted critical point. In the paramagnet, $[\langle q^{2}\rangle]$ sits near $1/N$.


## How $J$ is sampled

The word used earlier for $\sigma$ was "width". It means the standard deviation of $J_b$. It is not a linear size, and it is not a second random source on top of $J$.

For each pair $(L, \mathrm{sample\_id})$ the drawer in `kagome_qea.sample.draw_couplings` writes one independent standard normal on every stored bond,

$$
z_b \sim \mathcal{N}(0, 1).
$$

The generator is PCG64, seeded by `SeedSequence` with entropy `[1000, L, sample_id, 1]`. The stored array is these $z_b$. The ensemble name is `gaussian_nn`. The mean is zero: there is no fixed ferromagnetic coupling underneath.

A run at standard deviation $\sigma$ uses

$$
J_b = \sigma z_b, \qquad h = 0,
$$

so $\mathrm{Var}(J_b) = \sigma^2$. The six values are $\sigma = 0.05, 0.1, 0.2, 0.5, 1, 2$, a factor of two apart, so the drops are evenly spaced on a logarithmic $\Gamma$ axis. `Hamiltonian.with_classical_scale(\sigma)` multiplies the stored $z_b$. The random numbers are not drawn again. At fixed $L$ and `sample_id`, every curve sees the same disorder pattern.

The entropy contains $L$, so sample 0 at $L = 4$ and sample 0 at $L = 9$ are different lists of bonds and different lists of $z_b$. Sizes do not share one disorder realization. Only $\sigma$ is a rescaling of a realization that was already stored.


## Friday grid

A periodic Kagome cell of any integer $L \ge 2$ closes. The torus has $3L^2$ sites, $6L^2$ bonds, and degree 4 at every site. Odd $L$ is allowed. $L = 2$ ($N = 12$) stays the exact check, not a point on the figure.

The figure is one size and six standard deviations:

$$
L = 8, \qquad N = 192, \qquad \sigma = 0.05, 0.1, 0.2, 0.5, 1, 2.
$$

That size has its own 32 draws, `sample_id = 0, ..., 31`. The same stored $z_b$ is multiplied by $\sigma$. That is $6 \times 32 \times 7 = 1344$ projector runs. The $L = 6$ trial is nine runs on sample 0 only: $\sigma = 0.05, 1, 2$ and $\Gamma/\sigma = 1, 2, 3$.

- Dimensionless grid: $\Gamma/\sigma = 0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5$. On the $\Gamma$ axis, $\sigma = 2$ runs from $1$ to $7$ and $\sigma = 0.05$ runs from $0.025$ to $0.175$.
- Population: 5000 walkers, 40 discarded blocks, 24 forward cycles. At $\sigma = 1$ the block is $\tau_{\mathrm{block}} = 0.2$ and $\tau_{\mathrm{forward}} = 4$. Those settings matched the exact $L = 2$ overlap. At a general $\sigma$ the times are $\tau_{\mathrm{block}} = 0.2/\sigma$ and $\tau_{\mathrm{forward}} = 4/\sigma$. The number of blocks stays the same. Because $H(\sigma, \Gamma) = \sigma\, H(1, \Gamma/\sigma)$, a time $\tau$ at standard deviation $\sigma$ is the same projection as a time $\sigma\tau$ at standard deviation $1$. Leaving the times fixed would under-project every curve below $\sigma = 1$.

$L = 3, 5, 7, 9, 10$, draws past 32, and a finer step in $\Gamma$ are the work that follows the saved figure. They do not start while a Friday point is still missing.


In [ ]:
import itertools

FRIDAY_SIZES = (8,)
TRIAL = (6, (0.05, 1.0, 2.0), (1.0, 2.0, 3.0))
LATER_SIZES = (3, 4, 5, 6, 7, 9, 10)
SIGMAS = (0.05, 0.1, 0.2, 0.5, 1.0, 2.0)
RATIOS = (0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 3.5)
N_SAMPLES = 32

friday = list(itertools.product(FRIDAY_SIZES, range(N_SAMPLES), SIGMAS, RATIOS))
print(f"{len(friday)} projector runs in the Friday figure")
for size in FRIDAY_SIZES:
    n_runs = sum(job[0] == size for job in friday)
    print(f"L={size:<2}  sites {3 * size * size:<3}  runs {n_runs}")
for sigma in SIGMAS:
    gammas = tuple(round(sigma * ratio, 4) for ratio in RATIOS)
    print(f"sigma {sigma:<4}  Gamma {gammas}")
print(f"L=6 trial: sigma {TRIAL[1]}  ratios {TRIAL[2]}  runs {len(TRIAL[1]) * len(TRIAL[2])}")
print(f"after the figure: L = {LATER_SIZES}")
print(
    "one H800 batch at fixed L:",
    N_SAMPLES * len(SIGMAS) * len(RATIOS),
    "populations",
)


## Parallel on the H800

Nothing in this notebook launches a job. The instance stays up.

Checked on 25 October 2026, the box has one GPU: an NVIDIA H800 with 80 GB, idle. There is no second device. Parallelism is many projector runs resident on that GPU together. The Friday figure does not run as a CPU farm.

`kagome_qea.pqmc.project` advances one walker at a time in Python and never calls CUDA. Production uses a batched copy of the same continuous-time process. The physics is unchanged: flat guide, escape rate $\lambda = 2 N \Gamma$, a hop that touches four bonds, forward-walked overlap, the same population control, and the same block length.

One run is one triple $(L, \mathrm{sample}, \sigma, \Gamma)$. Inside one $L$ every run has the same number of sites and the same bond list, so one launch holds the whole size:

$$
32 \times 6 \times 7 = 1344
$$

populations of 5000 walkers. At $L = 8$ the spins are a few gigabytes. Weights, diagonal energies, and the per-population couplings fit beside them in the 80 GB. Walkers take their hops together until the block time is reached. Each population then resamples by itself. $\Gamma$ and $J_b = \sigma z_b$ are arrays with one entry per population.

The $L = 6$ trial uses the Python projector on one sample and nine pairs. It is read before the $L = 8$ batch is launched. The batched code is used for the figure only after it matches `exact.ground_state` at $L = 2$ for $\sigma = 1$, $\Gamma = 1$ and for $\sigma = 0.05$, $\Gamma = 0.05$.

The first $L = 8$ population is a probe of `min_unique_ancestors`. A handful of ancestral lines means the population is raised and $L = 8$ is repeated before those points go on the figure. The 32 draws and the forward walk stay.

Finished files are skipped. The batch writes

```text
data/kagome/gaussian_nn/pbc/L{L}/pqmc/h0/sigma{sigma}/gamma{gamma}/sample{id}_seed0.npz
```

under `/root/autodl-tmp/kagome-qea/` on the instance, and the same relative path after the copy back. A file that is already there is skipped.


## Figure

The Friday figure is the $L = 8$ torus, two panels.

- $[\langle q^{2}\rangle]$ and $[B]$ against $\Gamma$, one line for each $\sigma = 0.05, 0.1, 0.2, 0.5, 1, 2$, with the jackknife error. This is the shift.
- The same points against $\Gamma/\sigma$. The six lines lie on one curve. A $1/N$ line is drawn on the overlap panel.
